In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
from sklearn.metrics import classification_report

In [ ]:
# Load predictions
predictions_df = pd.read_csv(snakemake.input.predictions)
print(f"Loaded predictions with shape: {predictions_df.shape}")
print(f"Columns: {list(predictions_df.columns)}")

In [ ]:
# Extract ground truth and predictions
if 'ground_truth' in predictions_df.columns and 'predicted_class' in predictions_df.columns:
    y_true = predictions_df['ground_truth']
    y_pred = predictions_df['predicted_class']
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted')
    
    print(f"Accuracy: {accuracy:.3f}")
    print(f"F1-score (weighted): {f1:.3f}")
    
    # Generate confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    unique_labels = sorted(list(set(y_true) | set(y_pred)))
    
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred))
    
else:
    print("No ground truth or predictions found")
    accuracy = 0.0
    f1 = 0.0
    cm = np.array([[0]])
    unique_labels = ['unknown']

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=unique_labels, yticklabels=unique_labels)
plt.title(f'Confusion Matrix - {snakemake.wildcards.dataset}\n{snakemake.wildcards.metadata_col}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()

# Save plot
Path(snakemake.output.confusion_matrix).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(snakemake.output.confusion_matrix, dpi=300, bbox_inches='tight')
plt.close()

print(f"Saved confusion matrix to {snakemake.output.confusion_matrix}")

In [ ]:
# Save performance metrics
metrics = {
    'dataset': snakemake.wildcards.dataset,
    'metadata_col': snakemake.wildcards.metadata_col,
    'grouping': snakemake.wildcards.grouping,
    'accuracy': float(accuracy),
    'f1_weighted': float(f1),
    'n_samples': len(predictions_df),
    'n_classes': len(unique_labels)
}

Path(snakemake.output.performance_metrics).parent.mkdir(parents=True, exist_ok=True)
with open(snakemake.output.performance_metrics, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"Saved performance metrics to {snakemake.output.performance_metrics}")
print(f"Metrics: {metrics}")